In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from datetime import datetime
from utils import *

In [3]:
files = sorted(glob.glob('C:\\Users\ikiru\data\solo\phi\polar/*'))
files_ = sorted(glob.glob('C:\\Users\ikiru\data\sdo\hmi\polar/*'))
print(len(files))
print(files[0], files[-1])

275
C:\Users\ikiru\data\solo\phi\polar\solo_L2_phi-fdt-blos_20250426T000003_V202601302305_0544260501.fits.gz C:\Users\ikiru\data\solo\phi\polar\solo_L2_phi-fdt-blos_20250531T182003_V202601302340_0545310507.fits.gz


In [12]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 450#512

grid = np.mgrid[:nx,:ny].astype(np.float32)
view_new = View(nx, ny, xc, yc, Rsun, 0, 0, 0)

plt.ioff()

for i, file in enumerate(files[:1]):
    print(file)

    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    view = View.from_header(header)
    view_new = view_new.update(crln=view.crln, crlt=view.crlt)#, rsun=view.rsun)

    transform = view_new.to_synoptic(correct_mu=True, mu_thr=0.1) - view.to_synoptic(correct_mu=True, correct_dr=False, mu_thr=0.1)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    grid_, _ = (~view_new.to_synoptic())(np.mgrid[-90:90:1, -180:180:1])
    grid_ = np.array(grid_)
    meridians = grid_[:,:,::15]
    parallels = np.transpose(grid_[:,::15,:], (0,2,1))

    fig, ax = plt.subplots(figsize=(10,10))
    im = ax.imshow(data, 'seismic', vmin=-100, vmax=100, origin='lower')

    ax.plot(meridians[1], meridians[0], color='black', ls='--', lw=0.5)#, alpha=0.5)
    ax.plot(parallels[1], parallels[0], color='black', ls='--', lw=0.5)#, alpha=0.5)

    cax = ax.inset_axes((0.8, 0.985, 0.2, 0.015))
    fig.colorbar(im, cax=cax, orientation='horizontal', label=r'$B_{los}$, G')
    ax.axis('off')

    ax.annotate(datetime.fromisoformat(header['DATE-OBS']).replace(microsecond=0), (0.7,0.05), size=16, xycoords='axes fraction')
    plt.tight_layout()

    file_ = file.split("\\")[-1].split(".")[2]
    plt.savefig(f'temp/{file_}.png')
    #plt.savefig(f'temp/{file.split("/")[-1].split(".")[2]}.png')
    plt.close()

plt.ion()

D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_000000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_020000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_040000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_060000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_080000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_100000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_120000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_140000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_160000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_180000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_200000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241101_220000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\hmi.M_720s.20241102_000000_TAI.3.magnetogram.fits
D:/data/sdo/hmi/nonpolar\

In [8]:
file.split('\\')

['D:/data/solo/phi/nonpolar',
 'solo_L2_phi-fdt-blos_20241030T001503_V202602220112_0450300501.fits.gz']